In [ ]:
import sys

sys.path.append("../modelling")
# Import necessary libraries
import pandas as pd # For data manipulation and analysis
import numpy as np # For numerical operations
import pickle # For serializing and deserializing Python objects
from modelling_utils import train_knn_classifier, evaluate_model, plot_model_comparison, extract_model_features, compute_knn_indices, compute_overlap_matrix, plot_overlap_matrix, split_shuffle_data, apply_aggregation_filter
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import cohen_kappa_score


In [ ]:
# Load features from a pickle file
feature_dict_path = "/home/suraj/Repositories/FM-extractors-radiomics/evaluation/features/nsclc_radiomics_stability.pkl" # Path to the pickle file containing features
with open(feature_dict_path, 'rb') as file: # Open the file in read binary mode
    data = pickle.load(file) # Load the data from the pickle file

In [ ]:
# Store test accuracies for each model
results_list = []

# Iterate through each model's features
for model_name, values in data.items(): # Loop through each model and its corresponding features

    trial_0_data = [v for v in values["train"] if v["row"]["trial"] == 0]
    labels = [v["row"]["survival"] for v in trial_0_data] # Extract survival labels for the training set
    
    # Stack features
    items = np.vstack([apply_aggregation_filter(v["feature"], model_name) for v in trial_0_data]) # Stack features for the training set

    # train_items, val_items, train_labels, val_labels = train_test_split(items, labels, train_size=0.6)
    best_model, study = train_knn_classifier(items, labels, items, labels)
    # Bin the continuous predictions into many discrete categories to mimic a continuous kappa.
    # Here we use 100 bins over the interval [0, 1].      
    preds = best_model.predict_proba(items)[:, 1]
    bins = np.linspace(0, 1, 101)
    binned_preds = np.digitize(preds, bins) - 1

    for n_trial in range(1, 50):
        trial_n_data = [v for v in values["train"] if v["row"]["trial"] == n_trial]
        labels = [v["row"]["survival"] for v in trial_n_data] # Extract survival labels for the training set
        items = np.vstack([apply_aggregation_filter(v["feature"], model_name) for v in trial_n_data]) # Stack features for the training set
        trial_preds = best_model.predict_proba(items)[:, 1]

        binned_trial_preds = np.digitize(trial_preds, bins) - 1
        # Calculate the weighted Cohen's kappa score using linear weights as a continuous approximation.
        score = cohen_kappa_score(binned_preds, binned_trial_preds, weights="linear")
        row = {
            "Model": model_name,
            "Trial": n_trial,
            "AUC": score
        }
        print(row)
        results_list.append(row)

In [ ]:
results_df = pd.DataFrame(results_list)
results_df

In [ ]:
import plotly.express as px

# Create a box plot without mapping colors to "Model" to ensure all boxes use the same color.
fig = px.box(results_df, x="Model", y="AUC",
             title="AUC Distribution Across Trials by Model",
             template="simple_white")

# Update traces to ensure every box has the same fill color.
fig.update_traces(marker=dict(color="#1f77b4", size=8, line=dict(width=1, color="black")),
                  boxmean=True)

fig.show()
